# 3.2.3 SASRec 序列召回

> 阅读版与 Web 应用内容一致；实验数值来自本 Notebook 的已执行输出。

## Goal

如何用因果自注意力同时建模近期转移与较长兴趣，并预测用户下一部可能喜欢的电影？

## Setup

本 Notebook 的默认真实数据是 **MovieLens 1M：按 SASRec 论文协议执行 leave-two-out 与 100 负例评测**。`smoke` 档读取仓库内可审计的确定性切片，`full` 档扩大到官方完整文件；两档都不制造交互、曝光、标签或行为序列。切片规则、源地址、哈希与许可记录在 `data/README.md` 及对应数据目录。

**主要资料：** [Kang & McAuley, 2018, SASRec](https://arxiv.org/abs/1808.09781) · [作者官方实现](https://github.com/kang205/SASRec)

In [ ]:
from pathlib import Path
import os, sys, json
import torch
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.environ.setdefault("RECSYS_PROFILE", "full")
PROFILE = os.environ["RECSYS_PROFILE"]
CUDA_AVAILABLE = torch.cuda.is_available()
from recsys_lab.data import (load_movielens, movielens_provenance, load_amazon_2023,
                             amazon_provenance, load_kuairand, kuairand_provenance,
                             load_amazon_2018, amazon_2018_provenance,
                             load_movielens_1m, movielens_1m_provenance,
                             load_mind_amazon_books, mind_amazon_provenance,
                             load_census_income, census_income_provenance)
DATASET_KEY = "movielens-1m"
if DATASET_KEY == "movielens":
    real_ratings, real_movies = load_movielens()
    real_interactions = real_ratings
    REAL_DATASET = movielens_provenance(real_ratings)
elif DATASET_KEY == "amazon-books":
    real_ratings = load_amazon_2018("Books") if PROFILE == "full" else load_amazon_2023()
    real_interactions, real_movies = real_ratings, None
    REAL_DATASET = amazon_2018_provenance(real_ratings) if PROFILE == "full" else amazon_provenance(real_ratings)
elif DATASET_KEY == "mind-amazon-books":
    real_ratings = load_mind_amazon_books() if PROFILE == "full" else load_amazon_2023()
    real_interactions, real_movies = real_ratings, None
    REAL_DATASET = mind_amazon_provenance(real_ratings) if PROFILE == "full" else amazon_provenance(real_ratings)
elif DATASET_KEY == "amazon-electronics":
    real_ratings = load_amazon_2018("Electronics") if PROFILE == "full" else load_amazon_2023()
    real_interactions, real_movies = real_ratings, None
    REAL_DATASET = amazon_2018_provenance(real_ratings) if PROFILE == "full" else amazon_provenance(real_ratings)
elif DATASET_KEY == "movielens-1m":
    real_ratings = load_movielens_1m() if PROFILE == "full" else load_amazon_2023()
    real_interactions, real_movies = real_ratings, None
    REAL_DATASET = movielens_1m_provenance(real_ratings) if PROFILE == "full" else amazon_provenance(real_ratings)
elif DATASET_KEY == "census-income" and PROFILE == "full":
    census_train_x, census_train_y, census_test_x, census_test_y = load_census_income()
    real_interactions, real_movies, real_ratings = census_train_x, None, census_train_x
    REAL_DATASET = census_income_provenance()
elif DATASET_KEY == "amazon-2023":
    real_ratings = load_amazon_2023()
    real_interactions, real_movies = real_ratings, None
    REAL_DATASET = amazon_provenance(real_ratings)
else:
    real_interactions, real_movies = load_kuairand()
    real_ratings = real_interactions
    REAL_DATASET = kuairand_provenance(real_interactions)
print({"profile": PROFILE, "root": str(ROOT), "real_dataset": REAL_DATASET,
       "cuda_available": CUDA_AVAILABLE,
       "cuda_device": torch.cuda.get_device_name(0) if CUDA_AVAILABLE else None})
assert REAL_DATASET["randomly_fabricated_rows"] == 0

## 学习地图

1. 从原始论文理解系统约束；
2. 用可手算数字读懂公式和形状；
3. 检查数据、切分与标签；
4. 使用工业框架模型类训练；
5. 分开验证训练、推理和测试；
6. 用实际输出讨论失败边界。

**本节问题：** 如何用因果自注意力同时建模近期转移与较长兴趣，并预测用户下一部可能喜欢的电影？

**先修知识：** 3.0 的向量、概率与损失函数。第一次阅读无需推导梯度，只要能解释输入、输出和形状。

## Paper & Context

SASRec 用单向 Transformer 编码按时间排列的行为，并在每个位置预测下一物品。它在稀疏数据上可以聚焦少数近期行为，在较密数据上也能利用更长依赖；相对 RNN，训练可并行，但注意力成本随序列长度平方增长。

**来源：** [Kang & McAuley, 2018, SASRec](https://arxiv.org/abs/1808.09781) · [作者官方实现](https://github.com/kang205/SASRec)

### 原文实验设计与关键结论

原文在 Amazon Beauty/Games、Steam、MovieLens 四个数据集上按时间做 next-item 评测。full 档使用论文公开的 MovieLens-1M、leave-two-out、101-item sampled ranking 和模型超参数；smoke 不进入论文数值比较。

请区分三层证据：论文中的离线实验、本 Notebook 验证的代码链路、生产系统尚需验证的在线收益。三者不能互相替代。

## Reproduction Contract

**正式数据：** MovieLens 1M  
**资源 ID：** `movielens-1m-full`  
**切分：** paper protocol: leave last for test and penultimate for validation; every observed rating is implicit feedback  
**指标：** HitRate@10, NDCG@10  
**与论文比较边界：** compare with SASRec Table III only in full profile

`full` 是论文对照唯一有效的效果模式：它不截断用户、物品或测试行。`smoke` 只做张量、损失和推理链路回归，不进入论文数值比较。

## Model Structure & Formula Walkthrough

![Figure 1 · SASRec training and inference](/static/paper-figures/sasrec.webp)

> **论文原图节选** · Figure 1 · SASRec training and inference · PDF p.1。下图直接截取自原文，用于对照下方公式与代码。

### 关键模块

- **物品 + 位置 Embedding**：物品向量说明“看了什么”，位置向量说明“在序列中何时看到”，两者相加作为输入。
- **因果自注意力块**：自注意力计算每对行为之间的相关性；因果遮罩把未来位置权重设为 0；可堆叠多个 block，配合残差与 LayerNorm（见 3.0A §5.1）。
- **Next-item 预测头**：取最后有效位置的输出与物品库做点积，用 softplus pairwise 损失使正样本打分高于负样本，用于 Top-K 召回。

### 结构：只看过去的 Transformer

第 $t$ 个输入是商品与位置 embedding 之和 $x_t=e_{i_t}+p_t$。线性投影得到 $Q=XW_Q,K=XW_K,V=XW_V$，再计算

$$A=\mathrm{softmax}\left(\frac{QK^\top}{\sqrt d}+M\right),\qquad H=AV.$$

因果 mask $M_{t,j}$ 在 $j>t$ 时为 $-\infty$，softmax 后未来权重变成 0。除以 $\sqrt d$ 是为了避免维度变大时点积过大、softmax 过早饱和。注意力后还有残差连接、LayerNorm 和逐位置前馈网络。最后用位置 $t$ 的表示区分下一件正商品与负商品：

$$L_t=\mathrm{softplus}(-h_t^\top e^+)+\mathrm{softplus}(h_t^\top e^-).$$

训练时所有位置可并行，推理时取最后有效位置做全库点积或 ANN。

### 公式到代码

`run_sasrec` 从真实时间戳生成序列、正目标和负目标；Torch-RecHub SASRec 实现 item/position embedding、causal attention 与 pairwise loss。

阅读源码时按“张量形状 → 前向计算 → score → loss → metric”五步追踪，不需要一次读完整个工程文件。

## Math by Hand

把 item embedding 与位置 embedding 相加得到 $X$。单头注意力为 $\mathrm{softmax}(QK^\top/\sqrt d)V$，其中因果 mask 把未来位置设为 $-\infty$。位置 $t$ 的表示只能读取 $i_1,\ldots,i_t$，再与正/负物品向量做点积并优化 pairwise logistic loss。

下面用 NumPy/Matplotlib 验证直觉。二维图只是教学投影，工业 embedding 虽有更多维，计算规则相同。

In [ ]:
import numpy as np, matplotlib.pyplot as plt
length=6
logits=np.fromfunction(lambda row,col: 2.0-np.abs(row-col)*.55,(length,length))
logits[np.triu(np.ones((length,length),dtype=bool),1)]=-np.inf
weights=np.exp(logits-np.max(logits,axis=1,keepdims=True)); weights/=weights.sum(1,keepdims=True)
fig,ax=plt.subplots(figsize=(5.4,4.2)); image=ax.imshow(weights,cmap='YlGn'); ax.set(title='SASRec causal attention',xlabel='history position',ylabel='prediction position'); plt.colorbar(image,ax=ax); plt.show()
print('row sums =',weights.sum(1).round(3))

## Data

full 档使用完整 MovieLens 1M：所有观察评分都作为隐式交互，最后一项测试、倒数第二项验证，训练负例只从用户从未交互的目录物品中采样；评测为 1 个真值加 100 个未观察负例。

**防泄漏清单：**按时间切分；item 映射只表达已知目录，不读取测试标签；低评分或未点击负反馈均来自数据中的已观察行；序列只看预测时刻以前；测试集只在最后评价。CPU 档使用真实数据的确定性子集，**不是统一 benchmark 成绩**。

## Model & Framework

实际使用 torch_rechub.models.matching.SASRec，包含 item/position embedding、causal multi-head attention 与 pairwise loss；线上可分离用户表示和 item 向量。

smoke 档强调模型类、张量契约和指标链路真实可运行；full 档应替换原始数据、分布式配置、索引/服务和资源预算，而不是只增加 epoch。

In [ ]:
import inspect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from importlib import import_module
from recsys_lab.runtime import save_records

# 算法实现就在当前章节目录，不再通过公共模块隐藏。
chapter_train = import_module("chapter_code.3_2_3_sasrec.train")
run_sasrec = chapter_train.run_sasrec

print("实际执行函数源码（包含数据、训练、推理和测试）：")
print(inspect.getsource(run_sasrec))

## Train & Inference

下一格固定 seed、构造数据、实例化模型、训练并进入推理路径。生成式章节在 CUDA 上执行完整评测；CPU 环境只验证缩小后的基本张量与约束链路。

In [ ]:
result = run_sasrec()
print({'framework': result['framework'], 'dataset': result.get('dataset', {}),
       'device': result.get('device'), 'validation_mode': result.get('validation_mode')})
print('inference contract:', '截取最近 L 个真实行为 → 因果 Transformer → 最后位置用户向量 → 全库点积/ANN Top-K；屏蔽已见物品并监控序列截断与 P99。')
assert np.isfinite(result['loss_curve']).all()
print('loss:', round(result['loss_curve'][0],4), '→', round(result['loss_curve'][-1],4))

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(10.5,3.5))
axes[0].plot(result['loss_curve'],color='#4f8f00',lw=2); axes[0].set(title='Training loss',xlabel='epoch',ylabel='loss'); axes[0].grid(alpha=.2)
metrics={'hr@10': result['hr@10'], 'popularity_hr@10': result['popularity_hr@10']}
axes[1].bar(range(len(metrics)),list(metrics.values()),color=['#7ca832','#e1874b','#6d88a4'][:len(metrics)])
axes[1].set_xticks(range(len(metrics)),list(metrics),rotation=18); axes[1].set(title='Executed test metrics',ylim=(0,max(1.0,max(metrics.values())*1.15)))
for index,value in enumerate(metrics.values()): axes[1].text(index,value,f'{value:.3f}',ha='center',va='bottom')
plt.tight_layout(); plt.show(); display(pd.Series(metrics,name='value').to_frame())

In [ ]:
# 论文数字只能在数据、切分、候选和指标全部同口径时相减。
paper_protocol = json.loads('{"dataset": "MovieLens 1M", "resource": "movielens-1m-full", "filters": "paper-defined iterative user/item 5-core; expected 6,040 users, 3,416 items, 999,611 actions", "split": "paper protocol: leave last for test and penultimate for validation; every observed rating is implicit feedback", "metrics": ["HitRate@10", "NDCG@10"], "candidate_protocol": "1 positive + 100 sampled unobserved items", "model_protocol": "2 blocks, max length 200, embedding dimension 50, one attention head, dropout 0.2, Adam lr=0.001", "paper_targets": {"HitRate@10": 0.8245, "NDCG@10": 0.5905}, "paper_comparison": "compare with SASRec Table III only in full profile"}')
paper_targets = paper_protocol.get('paper_targets', {})
metric_key = {'HitRate@10':'paper_protocol_hr@10', 'NDCG@10':'paper_protocol_ndcg@10',
              'AUC':'auc', 'LogLoss':'logloss'}
dataset_name = result.get('dataset', {}).get('dataset', '')
dataset_aligned = paper_protocol.get('dataset', '').split(',')[0].casefold() in dataset_name.casefold()
comparison_eligible = PROFILE == 'full' and dataset_aligned
rows=[]
for paper_metric,target in paper_targets.items():
    result_key=metric_key.get(paper_metric)
    value=result.get(result_key) if result_key else None
    rows.append({'metric':paper_metric,'tutorial':value,'paper':target,
                 'absolute_gap':None if value is None or not comparison_eligible else float(value)-float(target),
                 'comparable':comparison_eligible and value is not None})
if rows:
    display(pd.DataFrame(rows))
    if not comparison_eligible:
        print('NOT COMPARABLE：当前运行的数据/协议与论文不完全一致，不计算复现差值。')
else:
    print('论文没有可公开、可同口径复现的绝对目标；本节只报告结构与公开协议验证。')

## Test & Results Discussion

In [ ]:
display(Markdown(f'''### 本次已执行结果

- 主指标 hr@10 = **{result['hr@10']:.4f}**。
- 辅助指标 popularity_hr@10 = **{result['popularity_hr@10']:.4f}**。
- 本节没有把不同任务的数值伪装成 baseline；章节总结只做同口径比较。
- 训练损失从 **{result['loss_curve'][0]:.4f}** 降到 **{result['loss_curve'][-1]:.4f}**。损失下降只说明优化工作，不等于泛化或业务收益。
- **结果解释：** full 档按论文使用 2 blocks、d=50、长度 200、dropout 0.2 与 lr=0.001；只有该档可与 Table III 的 ML-1M HR/NDCG 对照。

### 工业边界

截取最近 L 个真实行为 → 因果 Transformer → 最后位置用户向量 → 全库点积/ANN Top-K；屏蔽已见物品并监控序列截断与 P99。

上线前还需验证延迟、吞吐、内存/显存、数据新鲜度、校准、回滚和线上 A/B。
'''))

In [ ]:
record={
    'algorithm': 'SASRec 序列召回',
    'primary_metric': 'hr@10', 'primary_value': float(result['hr@10']),
    'secondary_metric': 'popularity_hr@10', 'secondary_value': float(result['popularity_hr@10']),
    'baseline_metric': None,
    'baseline_value': float(result[None]) if False else None,
    'framework': result['framework'], 'source_notebook': '3_2_3_sasrec',
    'validation_mode': result.get('validation_mode', 'standard'),
    'dataset': result['dataset']['dataset'],
    'randomly_fabricated_rows': int(result['dataset']['randomly_fabricated_rows'])
}
path=save_records('chapter_3_2','3_2_3_sasrec',[record]); print('saved:',path.relative_to(ROOT))

## Checks

自动断言用于防止数据、训练和指标链路静默失效，不是效果证明。

In [ ]:
assert result['loss_curve'][-1] < result['loss_curve'][0]
assert 0 <= float(result['hr@10']) <= 1
assert np.isfinite(float(result['popularity_hr@10']))
print('PASS：数据、训练、推理、测试和结果产物均已验证。')

## Next Steps

1. 换成对应公开数据的完整时间切分；2. 增加强 baseline 与消融；3. 记录效果、延迟和成本；4. 映射到 TorchEasyRec/官方 full profile；5. 只在相同候选与数据口径下比较。